In [27]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [152]:
import torch
import torch.nn as nn
import math

class TransformerCls(nn.Module):

    def __init__(self, tokenizer, embedding_dim):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.tokenizer= tokenizer
        self.embedding = nn.Embedding(num_embeddings=tokenizer.get_vocab_size(with_added_tokens=True), embedding_dim=embedding_dim, padding_idx=1)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embedding_dim, nhead=8, batch_first=True)
        self.encoder =nn.TransformerEncoder(encoder_layer, num_layers=6)
        self.activation = nn.ReLU()
        self.linear = nn.Linear(embedding_dim, 3)
        self.softmax = nn.Softmax(dim=1)

        self.odd_dims = tuple(range(1, embedding_dim, 2))
        self.even_dims = tuple(range(0, embedding_dim, 2))

    def positional_encoding(self, x):

        seq_len, embedding_dim = x.shape[-2], x.shape[-1]

        position = torch.arange(seq_len, device=x.device).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, embedding_dim, 2, device=x.device)
            * (-math.log(10000.0) / embedding_dim)
        )

        pe = torch.zeros(seq_len, embedding_dim, device=x.device)

        pe[:, self.even_dims] = torch.sin(position * div_term)[:, : len(self.even_dims)]
        pe[:, self.odd_dims] = torch.cos(position * div_term)[:, : len(self.odd_dims)]
        return x + pe.to(dtype=x.dtype)

    def encode_text(self, x):
        if isinstance(x, str):
            x = [x]
        encodings = self.tokenizer.encode_batch(x)        # list of Encoding
       
        ids = [torch.tensor([2] + e.ids, dtype=torch.long) for e in encodings]
        ids = nn.utils.rnn.pad_sequence(ids, batch_first=True, padding_value=1)

        mask = ids != 1

        return ids, mask

    def forward(self, x):
        x, mask = self.encode_text(x)

        x = x.to(self.embedding.weight.device)
        mask = mask.to(self.embedding.weight.device)

        x = self.embedding(x)
        x = self.positional_encoding(x)
        x = self.encoder(x, src_key_padding_mask=~mask)
        x = self.activation(x)
        x = x[:,0,:]
        x = self.linear(x)
        return x

In [101]:
import pandas as pd 

class ReviewDataset(Dataset):
    def __init__(self, source_csv):
        self.data = pd.read_csv(source_csv)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, key):
        return self.data.iloc[key, 3], self.data.iloc[key, 2]

In [29]:
from google.colab import drive
drive.mount("/content/drive")

proj_path = '/content/drive/MyDrive/CSE 3521/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [91]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(proj_path + "modern/custom_tokenizer.json")

In [ ]:
training_set = ReviewDataset(proj_path + "data/train.csv")
val_set = ReviewDataset(proj_path + "data/val.csv")
test_set = ReviewDataset(proj_path + "data/test.csv")

training_loader = DataLoader(training_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16, shuffle=True)
test_loader = DataLoader(test_set, batch_size=16, shuffle=True)

In [151]:
d_model = 512
device = "cuda"
model = TransformerCls(tokenizer, 512)
model = model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(),lr = 0.001)

In [ ]:
def train_one_epoch(epoch):
    for i, data in enumerate(training_loader):
        running_loss = 0
        last_loss = 0

        inputs, labels = data
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        loss = loss_fn(outputs, labels)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        if i % 100 == 0:
            last_loss = running_loss / 1000 # loss per batch
            print(f'  batch {i} loss: {last_loss}')
            running_loss = 0
    return last_loss
            

In [ ]:
n_epoch = 5
best_vloss = 10000000

for epoch in range(n_epoch):
    print(f"for epoch {epoch}")

    model.train(True)
    avg_loss = train_one_epoch(epoch)

    running_vloss = 0
    model.eval()

    with torch.no_grad():
        for i, vdata in enumerate(val_loader):
            vinputs, vlabels = vdata
            vlabels = vlabels.to(device)
            voutputs = model(vinputs)
            vloss = loss_fn(voutputs, vlabels)
            running_vloss += vloss

    avg_vloss = running_vloss / (i + 1)
    print(f'train loss: {avg_loss} val loss: {avg_vloss}')

    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        torch.save(model.state_dict(), proj_path + "modern/model")

for epoch 0
  batch 1000 loss: 0.0009146534204483032
